# Three radiograph-grouped splits and the clip-bound sweep

This notebook produces Table 3 (seeds 42, 43 and 44), Tables 5 and 6, and Fig. 3.

The encoders are frozen, so each tooth's embedding does not depend on the split. Embeddings for all teeth are extracted once per encoder and cached in `_results_option1/emb_cache/`. For each split, the probe, temperature and residual head are refitted from the cached embeddings.

1. Three splits (seeds 42, 43, 44) at δ = 1.5: macro-AUC and macro-ECE of the reference and of the residual update, as mean ± standard deviation, and the maximum observed logit deviation.
2. Clip-bound sweep on seed 42 with δ ∈ {0.3, 0.5, 1.0, 1.5, 2.0, 3.0}: macro-AUC, macro-ECE and maximum logit deviation at each δ.

The first run extracts the Dental ViT-L embeddings at 512 × 512 pixels and takes longest; later runs load the cache. `04_tufts_shift_screening.ipynb` reads the same cache, so run this notebook first.

## 1. Environment and device

In [ ]:
import sys, subprocess
def pipq(*p): subprocess.run([sys.executable,"-m","pip","install","-q",*p], check=False)
# Packages are listed in requirements.txt. Uncomment to install them into the active kernel.
# pipq("torch","torchvision","scikit-learn","pillow","matplotlib","tqdm",
#      "huggingface_hub","open_clip_torch","timm","ml_collections")
import torch, numpy as np, os, glob, json, math
DEVICE="mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)


## 2. Data path

In [ ]:
DATA_DIR = "/path/to/DENTEX/training_data/quadrant-enumeration-disease"
assert os.path.isdir(DATA_DIR), f"Folder not found:\n{DATA_DIR}"

IMG_DIR=None
for root,_,files in os.walk(DATA_DIR):
    if os.path.basename(root).lower()=="xrays" and any(f.lower().endswith((".png",".jpg")) for f in files):
        IMG_DIR=root; break
if IMG_DIR is None:
    for root,_,files in os.walk(DATA_DIR):
        if any(s in root.lower() for s in ("_result","output","checkpoint")): continue
        if any(f.lower().endswith((".png",".jpg")) for f in files): IMG_DIR=root; break
assert IMG_DIR, "No xrays folder found."

ANN_PATH=None
for root,_,files in os.walk(DATA_DIR):
    for f in files:
        if f.endswith(".json"):
            try: j=json.load(open(os.path.join(root,f)))
            except Exception: continue
            if isinstance(j,dict) and "annotations" in j and any("category_id_3" in a for a in j.get("annotations",[])):
                ANN_PATH=os.path.join(root,f); break
    if ANN_PATH: break
assert ANN_PATH, "Diagnosis JSON not found."

OUTPUT_DIR=os.path.join(DATA_DIR,"_results_option1"); os.makedirs(OUTPUT_DIR,exist_ok=True)
CACHE_DIR=os.path.join(OUTPUT_DIR,"emb_cache"); os.makedirs(CACHE_DIR,exist_ok=True)
print("Images:",IMG_DIR,"\nLabels:",ANN_PATH,"\nCache:",CACHE_DIR)


## 3. Tooth-level targets for all teeth (the split is made later, per seed)

In [ ]:
coco=json.load(open(ANN_PATH))
cats3=sorted(coco["categories_3"],key=lambda c:c["id"])
DX_NAMES=[c["name"] for c in cats3]; NUM_DX=len(DX_NAMES)
id2idx={c["id"]:i for i,c in enumerate(cats3)}
img_file={im["id"]:im["file_name"] for im in coco["images"]}
tooth={}
for a in coco["annotations"]:
    key=(a["image_id"],a.get("category_id_1",-1),a.get("category_id_2",-1))
    if key not in tooth:
        tooth[key]={"image_id":a["image_id"],"file":img_file[a["image_id"]],"bbox":list(a["bbox"]),"y":np.zeros(NUM_DX,np.float32)}
    tooth[key]["y"][id2idx[a["category_id_3"]]]=1.0
    bx,by,bw,bh=tooth[key]["bbox"]; ax,ay,aw,ah=a["bbox"]
    x0,y0=min(bx,ax),min(by,ay); x1,y1=max(bx+bw,ax+aw),max(by+bh,ay+ah)
    tooth[key]["bbox"]=[x0,y0,x1-x0,y1-y0]
teeth=list(tooth.values())
Y_ALL=np.stack([t["y"] for t in teeth])
IMG_OF=np.array([t["image_id"] for t in teeth])
print(f"{len(teeth)} teeth | classes {DX_NAMES} | pos/class {Y_ALL.sum(0).astype(int).tolist()}")


## 4. Crop helpers

In [ ]:
from PIL import Image
PAD=0.15; _imc={}
def load_image(f):
    if f not in _imc: _imc[f]=Image.open(os.path.join(IMG_DIR,os.path.basename(f))).convert("RGB")
    return _imc[f]
def crop_tooth(img,bbox,pad=PAD):
    W,H=img.size; x,y,w,h=bbox; px,py=w*pad,h*pad
    x0=max(0,int(x-px));y0=max(0,int(y-py));x1=min(W,int(x+w+px));y1=min(H,int(y+h+py))
    return img if (x1<=x0 or y1<=y0) else img.crop((x0,y0,x1,y1))
def get_crop(i): t=teeth[i]; return crop_tooth(load_image(t["file"]),t["bbox"])
print("crop helper ready; example:", get_crop(0).size)


## 5. BiomedCLIP encoder

In [ ]:
import open_clip
from tqdm.auto import tqdm
def load_biomedclip():
    model,_,preprocess=open_clip.create_model_and_transforms("hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224")
    return model.visual.to(DEVICE).eval(), preprocess
@torch.no_grad()
def embed_all(model,preprocess,bs=32,l2norm=True):
    outs=[]; idxs=list(range(len(teeth)))
    for k in tqdm(range(0,len(idxs),bs),desc="embed",leave=False):
        ims=[preprocess(get_crop(i)) for i in idxs[k:k+bs]]
        x=torch.stack(ims).to(DEVICE); f=model(x)
        if isinstance(f,(tuple,list)): f=f[0]
        f=f.float().cpu()
        if l2norm: f=torch.nn.functional.normalize(f,dim=1)
        outs.append(f)
    return torch.cat(outs).numpy()
print("BiomedCLIP loader ready.")


## 6. Dental ViT-L encoder

In [ ]:
DENTAL_OK=False; dental_model=None; dental_preprocess=None
try:
    if not os.path.isdir("ViT-pytorch"):
        subprocess.run(["git","clone","--depth","1","https://github.com/jeonsworld/ViT-pytorch"],check=True)
    sys.path.insert(0,"ViT-pytorch")
    from models.modeling import VisionTransformer, CONFIGS
    from huggingface_hub import hf_hub_download
    ckpt=hf_hub_download(repo_id="Bryceee/Teeth_Visual_Experts_Models",
        filename="Teeth_Visual_Experts_ViT_L_periapical_images_cls_7diseases.bin")
    state=torch.load(ckpt,map_location="cpu")
    if isinstance(state,dict) and "state_dict" in state: state=state["state_dict"]
    pos_len=state["transformer.embeddings.position_embeddings"].shape[1]
    side=int(round(math.sqrt(pos_len-1))); DENTAL_IMG=side*16
    n_head=state["head.weight"].shape[0]
    print(f"inferred res {DENTAL_IMG}x{DENTAL_IMG} | head classes {n_head}")
    cfg=CONFIGS["ViT-L_16"]
    m=VisionTransformer(cfg,img_size=DENTAL_IMG,num_classes=n_head,zero_head=False)
    missing,unexpected=m.load_state_dict(state,strict=False)
    assert not [k for k in missing if k.startswith("transformer.")], "trunk keys missing"
    m=m.to(DEVICE).eval()
    import torchvision.transforms as T
    dental_preprocess=T.Compose([T.Resize((DENTAL_IMG,DENTAL_IMG)),T.ToTensor(),
                                 T.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])])
    class DentalEmbed(torch.nn.Module):
        def __init__(self,vit): super().__init__(); self.vit=vit
        @torch.no_grad()
        def forward(self,x): seq,_=self.vit.transformer(x); return seq[:,0]
    dental_model=DentalEmbed(m).to(DEVICE).eval()
    DENTAL_OK=True; print("dental ViT-L loaded")
except Exception as e:
    print("dental ViT-L not loaded:",repr(e),"\nContinuing with BiomedCLIP only.")


## 7. Embeddings for all teeth, extracted once and cached

The embeddings do not depend on the split, so they are computed once per encoder and saved as `.npy` files.

In [ ]:
def cached_embed(name, embed_fn):
    path=os.path.join(CACHE_DIR,f"emb_{name}.npy")
    if os.path.exists(path):
        arr=np.load(path); print(f"{name}: loaded cache {arr.shape}"); return arr
    print(f"{name}: extracting (first time, may take minutes)...")
    arr=embed_fn(); np.save(path,arr); print(f"{name}: cached {arr.shape}"); return arr

EMB_ALL={}
bmc_model,bmc_pre=load_biomedclip()
EMB_ALL["biomedclip"]=cached_embed("biomedclip", lambda: embed_all(bmc_model,bmc_pre,l2norm=True))
if DENTAL_OK:
    @torch.no_grad()
    def embed_all_dental(bs=16):
        outs=[]; idxs=list(range(len(teeth)))
        for k in tqdm(range(0,len(idxs),bs),desc="dental embed",leave=False):
            ims=[dental_preprocess(get_crop(i)) for i in idxs[k:k+bs]]
            x=torch.stack(ims).to(DEVICE)
            f=torch.nn.functional.normalize(dental_model(x).float().cpu(),dim=1)
            outs.append(f)
        return torch.cat(outs).numpy()
    EMB_ALL["dental_vitl"]=cached_embed("dental_vitl", embed_all_dental)
print("backbones cached:", list(EMB_ALL.keys()))


## 8. Reference probe, temperature, residual head and metrics

In [ ]:
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

def grouped_split(img_of, seed, vf=0.15, tf=0.15):
    ids=sorted(set(img_of.tolist())); rng=np.random.RandomState(seed); rng.shuffle(ids)
    n=len(ids); nte=int(n*tf); nva=int(n*vf)
    te=set(ids[:nte]); va=set(ids[nte:nte+nva]); tr_,va_,te_=[],[],[]
    for i,im in enumerate(img_of):
        (te_ if im in te else va_ if im in va else tr_).append(i)
    return np.array(tr_),np.array(va_),np.array(te_)

def fit_baseline(X,Yt):
    return [LogisticRegression(max_iter=1000,C=1.0).fit(X,Yt[:,c]) if len(np.unique(Yt[:,c]))>1 else None for c in range(NUM_DX)]
def baseline_logits(pr,X):
    o=np.zeros((len(X),NUM_DX),np.float32)
    for c,p in enumerate(pr): o[:,c]=p.decision_function(X) if p is not None else 0.0
    return o
def fit_temp(logits,y):
    T=nn.Parameter(torch.ones(1)); z=torch.tensor(logits); yt=torch.tensor(y)
    opt=torch.optim.LBFGS([T],lr=0.1,max_iter=60); bce=nn.BCEWithLogitsLoss()
    def cl(): opt.zero_grad(); l=bce(z/T.clamp(min=0.05),yt); l.backward(); return l
    opt.step(cl); return float(T.detach().clamp(min=0.05))
def ece(y,p,nb=10):
    b=np.linspace(0,1,nb+1); e=0.0
    for lo,hi in zip(b[:-1],b[1:]):
        m=(p>lo)&(p<=hi)
        if m.sum()==0: continue
        e+=(m.sum()/len(p))*abs(p[m].mean()-y[m].mean())
    return e
def macro_auc(yt,yp):
    v=[roc_auc_score(yt[:,c],yp[:,c]) for c in range(NUM_DX) if len(np.unique(yt[:,c]))==2]
    return float(np.mean(v)) if v else float("nan")
class EnvHead(nn.Module):
    def __init__(self,d,k,delta): super().__init__(); self.delta=delta; self.net=nn.Sequential(nn.Linear(d,256),nn.ReLU(),nn.Dropout(0.2),nn.Linear(256,k))
    def residual(self,x): return torch.clamp(self.net(x),-self.delta,self.delta)
    def forward(self,x,b): return b+self.residual(x)
def train_env(Xtr,btr,Ytr,Xva,bva,Yva,delta,epochs=60,lr=1e-3):
    h=EnvHead(Xtr.shape[1],NUM_DX,delta).to(DEVICE)
    opt=torch.optim.AdamW(h.parameters(),lr=lr,weight_decay=1e-4); crit=nn.BCEWithLogitsLoss()
    dl=DataLoader(TensorDataset(torch.tensor(Xtr),torch.tensor(btr),torch.tensor(Ytr)),batch_size=64,shuffle=True)
    best=1e9; bs=None
    for _ in range(epochs):
        h.train()
        for x,b,y in dl:
            x,b,y=x.to(DEVICE),b.to(DEVICE),y.to(DEVICE)
            opt.zero_grad(); crit(h(x,b),y).backward(); opt.step()
        h.eval()
        with torch.no_grad():
            vl=crit(h(torch.tensor(Xva).to(DEVICE),torch.tensor(bva).to(DEVICE)),torch.tensor(Yva).to(DEVICE)).item()
        if vl<best: best=vl; bs={k:v.detach().cpu().clone() for k,v in h.state_dict().items()}
    h.load_state_dict(bs); return h

def run_once(emb_all, seed, delta):
    tr,va,te=grouped_split(IMG_OF,seed)
    Xtr,Xva,Xte=emb_all[tr],emb_all[va],emb_all[te]
    Ytr,Yva,Yte=Y_ALL[tr],Y_ALL[va],Y_ALL[te]
    pr=fit_baseline(Xtr,Ytr); T=fit_temp(baseline_logits(pr,Xva),Yva)
    bte=baseline_logits(pr,Xte)/T; btr=baseline_logits(pr,Xtr)/T; bva=baseline_logits(pr,Xva)/T
    p_base=1/(1+np.exp(-bte))
    h=train_env(Xtr,btr,Ytr,Xva,bva,Yva,delta)
    with torch.no_grad(): final=h(torch.tensor(Xte).to(DEVICE),torch.tensor(bte).to(DEVICE)).cpu().numpy()
    p_env=1/(1+np.exp(-final)); dev=float(np.max(np.abs(final-bte)))
    return {"base_auc":macro_auc(Yte,p_base),"env_auc":macro_auc(Yte,p_env),
            "base_ece":float(np.mean([ece(Yte[:,c],p_base[:,c]) for c in range(NUM_DX)])),
            "env_ece":float(np.mean([ece(Yte[:,c],p_env[:,c]) for c in range(NUM_DX)])),
            "max_dev":dev,"guarantee":bool(dev<=delta+1e-4)}
print("machinery ready.")


## 9. Three splits: mean ± standard deviation (Table 3)

The cached embeddings are split three times by radiograph (seeds 42, 43, 44). The probe, temperature and residual head are refitted for each split.

In [ ]:
SEEDS=[42,43,44]; DELTA=1.5
crn={name:{s:run_once(EMB_ALL[name],s,DELTA) for s in SEEDS} for name in EMB_ALL}

def agg(name,key):
    vals=[crn[name][s][key] for s in SEEDS]; return float(np.mean(vals)),float(np.std(vals))

print(f"=== Three-split results (mean ± std over seeds {SEEDS}, δ={DELTA}) ===")
for name in EMB_ALL:
    print(f"\n{name}:")
    for key,label in [("base_auc","baseline AUC"),("env_auc","envelope AUC"),
                      ("base_ece","baseline ECE"),("env_ece","envelope ECE")]:
        m,s=agg(name,key); print(f"  {label:>13}: {m:.4f} ± {s:.4f}")
    g=all(crn[name][s]["guarantee"] for s in SEEDS); print(f"  bound holds on all seeds: {g}")


## 10. Clip-bound sweep (Tables 5 and 6, Fig. 3)

On seed 42, δ is varied and the macro-AUC, macro-ECE and maximum logit deviation of the residual update are recorded for both encoders. Dashed lines in the figure mark the reference values.

In [ ]:
import matplotlib.pyplot as plt
DELTAS=[0.3,0.5,1.0,1.5,2.0,3.0]; SWEEP_SEED=42
sweep={name:{d:run_once(EMB_ALL[name],SWEEP_SEED,d) for d in DELTAS} for name in EMB_ALL}

fig,(ax1,ax2)=plt.subplots(1,2,figsize=(12,4.2))
colors={"dental_vitl":"#1a3a5c","biomedclip":"#c9a227"}
for name in EMB_ALL:
    aucs=[sweep[name][d]["env_auc"] for d in DELTAS]
    eces=[sweep[name][d]["env_ece"] for d in DELTAS]
    base_auc=sweep[name][DELTAS[0]]["base_auc"]
    c=colors.get(name,"#444")
    ax1.plot(DELTAS,aucs,"o-",color=c,label=f"{name} (envelope)")
    ax1.axhline(base_auc,ls="--",color=c,alpha=0.5)
    ax2.plot(DELTAS,eces,"o-",color=c,label=name)
ax1.set_xlabel("δ (deviation bound)"); ax1.set_ylabel("envelope macro-AUC"); ax1.set_title("Utility vs δ\n(dashed = baseline AUC)"); ax1.legend(fontsize=8); ax1.grid(alpha=0.3)
ax2.set_xlabel("δ (deviation bound)"); ax2.set_ylabel("envelope macro-ECE"); ax2.set_title("Calibration vs δ"); ax2.legend(fontsize=8); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR,"delta_sweep.png"),dpi=140); plt.show()

# guarantee check across all δ
for name in EMB_ALL:
    ok=all(sweep[name][d]["guarantee"] for d in DELTAS)
    print(f"{name}: bound holds at every δ: {ok}")


## 11. Save results

In [ ]:
import datetime
out={
    "timestamp":datetime.datetime.now().isoformat(timespec="seconds"),
    "device":DEVICE,"seeds":SEEDS,"delta_default":DELTA,"deltas":DELTAS,
    "backbones":list(EMB_ALL.keys()),"class_names":DX_NAMES,
    "crn":{name:{"base_auc":agg(name,"base_auc"),"env_auc":agg(name,"env_auc"),
                 "base_ece":agg(name,"base_ece"),"env_ece":agg(name,"env_ece"),
                 "per_seed":crn[name]} for name in EMB_ALL},
    "delta_sweep":{name:{str(d):sweep[name][d] for d in DELTAS} for name in EMB_ALL},
}
p=os.path.join(OUTPUT_DIR,"option1_crn_deltasweep.json")
json.dump(out,open(p,"w"),indent=2)
print("Saved ->",p)
# compact table
print("\n=== Summary (mean ± std AUC / ECE) ===")
for name in EMB_ALL:
    ba,bas=agg(name,"base_auc"); ea,eas=agg(name,"env_auc")
    be,bes=agg(name,"base_ece"); ee,ees=agg(name,"env_ece")
    print(f"{name:>12}: AUC {ba:.3f}±{bas:.3f} -> {ea:.3f}±{eas:.3f} | ECE {be:.3f}±{bes:.3f} -> {ee:.3f}±{ees:.3f}")
